# HY-489 · Week 1 — Hands-on · **SOLUTIONS**

**Instructor / TA copy. Do not distribute before the session.**

Every answer is computed, not hard-coded, so this notebook stays correct if the
dataset is ever regenerated. Run it once before the lecture and read the output:
those are the numbers to expect on the projector.

Timing in the room: questions 1 to 3 take most pairs about six minutes, question 4
about five, and question 5 is where the discussion is. Do not let 1 to 3 run long.


In [ ]:
import pandas as pd
from pathlib import Path

LOCAL  = Path("../../lab01/data")
REMOTE = "https://raw.githubusercontent.com/kon0925/hy489/main/lab01/data/"
DATA = str(LOCAL) + "/" if LOCAL.exists() else REMOTE
print("reading from:", DATA)

patients   = pd.read_csv(DATA + "patients.csv",   parse_dates=["BIRTHDATE"])
conditions = pd.read_csv(DATA + "conditions.csv", parse_dates=["START"])
encounters = pd.read_csv(DATA + "encounters.csv", parse_dates=["START", "STOP"])
observations = pd.read_csv(DATA + "observations.csv", parse_dates=["DATE"])

print(patients.shape, conditions.shape, encounters.shape)


---
## 1 · Patient count and gender split

**Expect ~1,174 patients, roughly balanced.** Most students get this in one line.

Teaching point: `value_counts(normalize=True)` is the follow-up worth showing.


In [ ]:
print("patients:", len(patients))
print()
print(patients["GENDER"].value_counts())
print()
print(patients["GENDER"].value_counts(normalize=True).round(3))


---
## 2 · Five most common conditions

Teaching point, and the one to say out loud: this counts **descriptions**, which are
free text. Counting the `CODE` column instead is the correct habit, and the two
answers agree here only because the data is synthetic. In a real extract they do not.
This is the seed for Week 5.


In [ ]:
print(conditions["DESCRIPTION"].value_counts().head())
print()
# the same question asked properly, by code
by_code = (conditions.groupby(["CODE", "DESCRIPTION"]).size()
                     .sort_values(ascending=False).head())
print(by_code)
print()
print("distinct descriptions:", conditions["DESCRIPTION"].nunique(),
      "| distinct codes:", conditions["CODE"].nunique())


---
## 3 · Average encounters per patient

**Expect a mean of roughly 11.5.** Then push past the mean.

Teaching point: healthcare utilisation is heavily right-skewed. The mean describes
almost nobody. Ask the room what the median is before you print it.


In [ ]:
per_patient = encounters.groupby("PATIENT").size()

print("total encounters:", len(encounters))
print("mean per patient:  %.2f" % per_patient.mean())
print("median:           ", per_patient.median())
print("min / max:        ", per_patient.min(), "/", per_patient.max())
print()
print(per_patient.describe().round(2))
print()
# how much of the total is used by the busiest 10% of patients?
top = per_patient.sort_values(ascending=False)
n10 = int(len(top) * 0.10)
print(f"busiest 10% of patients account for {top.head(n10).sum() / top.sum():.1%} of all encounters")


---
## 4 · One patient's conditions in chronological order

Two things to draw out: the join on patient id, and the fact that a clinical
timeline is a *sort*, not a stored ordering. Nothing in the CSV is in time order.

Pick the patient programmatically so this works on any regeneration of the data.


In [ ]:
# pick a patient with a reasonable number of conditions
counts = conditions.groupby("PATIENT").size().sort_values(ascending=False)
pid = counts.index[0]

who = patients.loc[patients["Id"] == pid].iloc[0]
print("patient:", pid)
print("  born", who["BIRTHDATE"].date(), "|", who["GENDER"], "|", who["CITY"])
print("  conditions:", counts.iloc[0], "| encounters:", (encounters["PATIENT"] == pid).sum())
print()

timeline = (conditions.loc[conditions["PATIENT"] == pid, ["START", "CODE", "DESCRIPTION"]]
                      .sort_values("START"))
timeline


A fuller version, merging the patient in. This is the shape of every 'show me the
chart' screen ever built, and it previews Lab 1 Checkpoint 3.


In [ ]:
merged = conditions.merge(patients[["Id", "GENDER", "BIRTHDATE", "CITY"]],
                          left_on="PATIENT", right_on="Id", suffixes=("", "_pat"))
one = merged.loc[merged["PATIENT"] == pid].sort_values("START")

# age at onset - the kind of derived column students reach for next
one = one.assign(age_at_onset=((one["START"] - one["BIRTHDATE"]).dt.days / 365.25).round(1))
one[["START", "age_at_onset", "CODE", "DESCRIPTION"]]


---
## 5 · What looks too clean

**This is the point of the whole segment.** Collect three or four answers from the
room before showing anything.

What they should find:


In [ ]:
for name, df in [("patients", patients), ("encounters", encounters),
                 ("conditions", conditions), ("observations", observations)]:
    miss = df.isna().sum()
    miss = miss[miss > 0]
    print(f"{name:<14}{len(df):>7} rows | columns with missing values:",
          dict(miss) if len(miss) else "NONE")


In [ ]:
# no duplicate patients, ever
print("duplicate patient Ids:", patients["Id"].duplicated().sum())
print("duplicate names:      ", patients.duplicated(subset=["FIRST", "LAST", "BIRTHDATE"]).sum())
print()
# every condition is coded
print("conditions with no CODE:", conditions["CODE"].isna().sum())
print()
# one description per code, exactly
per_code = conditions.groupby("CODE")["DESCRIPTION"].nunique()
print("codes with more than one spelling of their description:", (per_code > 1).sum())


### The answers to accept

| What they say | Why it is right |
|---|---|
| No missing values anywhere | Real extracts are full of them. A column that is 40% empty is normal |
| Every condition has a code | In reality a large share arrive as free text with no code at all |
| One spelling per code | Real data has `Hypertension`, `HTN`, `hypertension `, `Essential HTN` |
| No duplicate patients | Registration desks create duplicates daily. Week 7 is about finding them |
| Dates all parse | Real files contain `00/00/0000`, dates in the future, and three formats in one column |
| Nobody has a typo in their name | Transliteration, accents, married names |

**The one exception in this dataset**, and worth showing if nobody finds it: the
`UNITS` column in `observations.csv` is empty for a few percent of rows. That is not
a bug. Those are the text-valued observations, which have no unit to record. This is
the difference between *missing* and *not applicable*, and it is the answer to Lab 1
Checkpoint 1 question 4.


In [ ]:
u = observations["UNITS"].isna()
print(f"observations with no UNITS: {u.sum()} of {len(observations)} ({u.mean():.1%})")
print()
print(observations.loc[u, ["DESCRIPTION", "VALUE", "UNITS"]].head(8).to_string(index=False))


---
## Debrief — the four points on the slide

1. **You already used HY-360.** `value_counts`, `groupby`, and a join on patient id.
   Health data is the relational model underneath.
2. **Meaning needs codes.** Question 2 counted free text. Ask what happens when two
   hospitals spell the same condition differently. That is Week 5.
3. **Synthetic is too tidy.** The cleanliness that makes this data safe to hand out
   is exactly what hides the problems the course exists to solve. Weeks 7 and 9.
4. **One patient, many tables.** Question 4 needed a filter and a sort across two
   files. A full record has to be reassembled. That is the integration problem, live.

### If the room is ahead of schedule

* Plot the age distribution. Does it look like a population?
* Count conditions per patient and compare with encounters per patient.
* Ask which of the columns in `patients.csv` would have to be removed before this
  file could leave a hospital. (`SSN`, `DRIVERS`, `PASSPORT`, `LAT`, `LON`, names,
  full birth dates.) That is Lab 5, and it costs ninety seconds to set up here.


In [ ]:
print(sorted(patients.columns))
